In [0]:
%pip install statsmodels

In [0]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

fc = (spark.table("workspace.vol_project.forecasts")
      .select("target_date", "model", "forecast", "actual").toPandas())

fc["qlike"] = fc["actual"] / fc["forecast"] - np.log(fc["actual"] / fc["forecast"]) - 1

L = fc.pivot(index="target_date", columns="model", values="qlike")
L.mean().sort_values()

In [0]:
def dm_test(a, b, lags=5):
    d = L[a] - L[b]
    res = sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return d.mean(), res.tvalues.iloc[0], res.pvalues.iloc[0]

pairs = [
    ("har_rolling_730", "har_expanding"),
    ("gjr_t_730",       "garch_t_730"),
    ("har_rolling_730", "gjr_t_730"),
    ("garch_t_730",     "naive"),
    ("har_rolling_730", "naive"),
    ("lgbm_har_730",  "har_rolling_730"),
("lgbm_full_730", "har_rolling_730"),
("har_wknd_730",  "har_rolling_730"),  
("harx_730",      "har_wknd_730"),
("lgbm_full_730", "harx_730"),          
]

pd.DataFrame(
    [(a, b, *dm_test(a, b)) for a, b in pairs],
    columns=["model_A", "model_B", "mean_loss_diff", "t_stat", "p_value"]
)